In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, current_date
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("SCD_Type2_Full_Pipeline") \
    .enableHiveSupport() \
    .getOrCreate()
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


# --------------------------------------------------
# 1️⃣ Create Existing Dimension Table (Warehouse)
# --------------------------------------------------

existing_data = [
    (1, "Ravi", "Salem", "2023-01-01", None, "Y"),
    (2, "Anu", "Chennai", "2023-01-01", None, "Y"),
    (3, "Kumar", "Madurai", "2023-01-01", None, "Y")
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("current_flag", StringType(), True)
])


dim_df = spark.createDataFrame(existing_data, schema)

dim_df.write.mode("overwrite").saveAsTable("customer_dim_scd2")

print("Initial Warehouse Table")
spark.table("customer_dim_scd2").show()

# --------------------------------------------------
# 2️⃣ New Incoming Data
# Ravi moved to Coimbatore
# Meena is new
# --------------------------------------------------

new_data = [
    (1, "Ravi", "Coimbatore"),  # Changed
    (2, "Anu", "Chennai"),      # Same
    (4, "Meena", "Trichy")      # New
]

new_columns = ["id", "name", "city"]

new_df = spark.createDataFrame(new_data, new_columns)


# --------------------------------------------------
# 8️⃣ Identify Unchanged Records
# These are records from the old dimension table that are still
# present in the new data and have not changed their attributes.
# This assumes 'join_df' is already defined from previous steps (e.g., cell 'ce93tC6mxNk2').
# --------------------------------------------------

unchanged_df = join_df.filter(
    (col("old.id").isNotNull()) &
    (col("new.id").isNotNull()) &
    (col("old.city") == col("new.city"))
).select(
    col("old.id").alias("id"),
    col("old.name").alias("name"),
    col("old.city").alias("city"),
    col("old.start_date").alias("start_date"),
    col("old.end_date").alias("end_date"),
    col("old.current_flag").alias("current_flag")
)

# --------------------------------------------------
# 9️⃣ Final Dimension Table
# --------------------------------------------------

final_df = unchanged_df.union(expired_df).union(new_records_df)

print("Final SCD Type 2 Table")
final_df.show()

# --------------------------------------------------
# 🔟 Overwrite Warehouse Table
# --------------------------------------------------

final_df.write.mode("overwrite").saveAsTable("customer_dim_scd2")

spark.table("customer_dim_scd2").show()

Initial Warehouse Table
+---+-----+-------+----------+--------+------------+
| id| name|   city|start_date|end_date|current_flag|
+---+-----+-------+----------+--------+------------+
|  2|  Anu|Chennai|2023-01-01|    NULL|           Y|
|  3|Kumar|Madurai|2023-01-01|    NULL|           Y|
|  1| Ravi|  Salem|2023-01-01|    NULL|           Y|
+---+-----+-------+----------+--------+------------+



NameError: name 'unchanged_df' is not defined

In [4]:

# --------------------------------------------------
# 3️⃣ Read Current Active Records
# --------------------------------------------------

old_df = spark.table("customer_dim_scd2") \
    .filter(col("current_flag") == "Y")



In [5]:
# --------------------------------------------------
# 4️⃣ Join New Data with Current Records
# --------------------------------------------------

join_df = old_df.alias("old").join(
    new_df.alias("new"),
    col("old.id") == col("new.id"),
    "fullouter"
)


In [6]:

# --------------------------------------------------
# 5️⃣ Identify Changed Records
# --------------------------------------------------

changed_df = join_df.filter(
    (col("old.city") != col("new.city")) &
    col("new.city").isNotNull()
)



In [ ]:

# --------------------------------------------------
# 6️⃣ Expire Old Records
# --------------------------------------------------

expired_df = changed_df.select(
    col("old.id"),
    col("old.name"),
    col("old.city"),
    col("old.start_date"),
    current_date().alias("end_date"),
    lit("N").alias("current_flag")
)


In [ ]:

# --------------------------------------------------
# 7️⃣ Insert New Records (for changed + new IDs)
# --------------------------------------------------

new_records_df = join_df.filter(
    col("new.id").isNotNull()
).select(
    col("new.id").alias("id"),
    col("new.name").alias("name"),
    col("new.city").alias("city"),
    current_date().alias("start_date"),
    lit(None).alias("end_date"),
    lit("Y").alias("current_flag")
)


In [1]:
# sde 3



from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder \
    .appName("SCD_Type3_Example") \
    .enableHiveSupport() \
    .getOrCreate()

# --------------------------------------------------
# 1️⃣ Existing Warehouse Table
# --------------------------------------------------

existing_data = [
    (1, "Ravi", "Salem", None),
    (2, "Anu", "Chennai", None)
]
# Define the schema explicitly with types
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("current_city", StringType(), True),
    StructField("previous_city", StringType(), True)
])

# columns = ["id", "name", "current_city", "previous_city"]

dim_df = spark.createDataFrame(existing_data, schema)

dim_df.write.mode("overwrite").saveAsTable("customer_dim_scd3")

print("Initial Table")
spark.table("customer_dim_scd3").show()

# --------------------------------------------------
# 2️⃣ New Incoming Data
# --------------------------------------------------

new_data = [
    (1, "Ravi", "Coimbatore"),  # changed
    (2, "Anu", "Chennai"),      # same
    (3, "Meena", "Trichy")      # new
]
new_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
])
# new_columns = ["id", "name", "city"]

new_df = spark.createDataFrame(new_data, new_schema)

# --------------------------------------------------
# 3️⃣ Read Existing Table
# --------------------------------------------------

old_df = spark.table("customer_dim_scd3")

# --------------------------------------------------
# 4️⃣ Join
# --------------------------------------------------

merged_df = old_df.alias("old").join(
    new_df.alias("new"),
    col("old.id") == col("new.id"),
    "fullouter"
)
# print("merfes")
merged_df.show()

# --------------------------------------------------
# 5️⃣ Apply SCD Type 3 Logic
# --------------------------------------------------

scd3_df = merged_df.select(
    coalesce(col("new.id"), col("old.id")).alias("id"),

    coalesce(col("new.name"), col("old.name")).alias("name"),

    # current_city logic
    when(col("old.id").isNull(), col("new.city"))  # new record
    .when(col("old.current_city") != col("new.city"), col("new.city"))  # changed
    .otherwise(col("old.current_city"))
    .alias("current_city"),

    # previous_city logic
    when(col("old.id").isNull(), None)  # new reco  rd
    .when(col("old.current_city") != col("new.city"), col("old.current_city"))  # changed
    .otherwise(col("old.previous_city"))
    .alias("previous_city")
)
scd3_df.show()

# --------------------------------------------------
# 6️⃣ Overwrite Table
# --------------------------------------------------

# Fix: Write to a temporary staging table, then use INSERT OVERWRITE
staging_table_name = "customer_dim_scd3_staging"
scd3_df.write.mode("overwrite").saveAsTable(staging_table_name)

spark.sql(f"INSERT OVERWRITE TABLE customer_dim_scd3 SELECT id, name, current_city, previous_city FROM {staging_table_name}")

# Clean up the staging table
spark.sql(f"DROP TABLE IF EXISTS {staging_table_name}")

spark.table("customer_dim_scd3").show()

Initial Table
+---+----+------------+-------------+
| id|name|current_city|previous_city|
+---+----+------------+-------------+
|  2| Anu|     Chennai|         NULL|
|  1|Ravi|       Salem|         NULL|
+---+----+------------+-------------+

+----+----+------------+-------------+---+-----+----------+
|  id|name|current_city|previous_city| id| name|      city|
+----+----+------------+-------------+---+-----+----------+
|   1|Ravi|       Salem|         NULL|  1| Ravi|Coimbatore|
|   2| Anu|     Chennai|         NULL|  2|  Anu|   Chennai|
|NULL|NULL|        NULL|         NULL|  3|Meena|    Trichy|
+----+----+------------+-------------+---+-----+----------+

+---+-----+------------+-------------+
| id| name|current_city|previous_city|
+---+-----+------------+-------------+
|  1| Ravi|  Coimbatore|        Salem|
|  2|  Anu|     Chennai|         NULL|
|  3|Meena|      Trichy|         NULL|
+---+-----+------------+-------------+

+---+-----+------------+-------------+
| id| name|current_city